### --- Day 6: Lanternfish ---

Puzzle description redacted as-per Advent of Code guidelines

You may find the puzzle description at: https://adventofcode.com/2021/day/6

In [2]:
#!import ../Utils.ipynb

In [3]:
var inputLines = LoadPuzzleInput(2021, 6);
WriteLines(inputLines, maxCols: 50);

Loading puzzle file: Day6.txt
Total lines: 1
Max line length: 599

2,5,3,4,4,5,3,2,3,3,2,2,4,2,5,4,1,1,4,4,5,1,2,1,5,


In [4]:
string[] testInputLines = [
    "3,4,3,1,2",
];

Ok, I think we can do this recursively. We know that...

* The current fish will produce a new fish after $n + 1$ days.
* It will continue producing fish every 7 days until time runs out.

So for each new fish we add, we can recursively add its contribution to the total.


In [5]:
int NewFish(int n, int daysRemain)
{
    if (daysRemain < 0) { return 0; }

    int result = 1;
    while (daysRemain > 0)
    {
        daysRemain -= (n + 1);
        n = 6;

        if (daysRemain >= 0)
        {
            result += NewFish(8, daysRemain);
        }
    }

    return result;
}

In [6]:
int CountAllFish(string[] inputLines, int daysRemain = 80)
{
    var nums = inputLines[0].ParseInts().ToArray();

    var result = nums.Select(n => NewFish(n, daysRemain)).Sum();
    return result;
}

In [7]:
// In this example, after 18 days, there are a total of 26 fish. After 80 days,
// there would be a total of 5934.

var testAnswer = CountAllFish(testInputLines);
Console.WriteLine(testAnswer);

5934


In [8]:
// Find a way to simulate lanternfish. How many lanternfish would there be after
// 80 days?

var part1Answer = CountAllFish(inputLines);
Console.WriteLine(part1Answer);

345387


In [9]:
// 345387 is correct!
Ensure(345387, part1Answer);

### --- Part Two ---

Puzzle description redacted as-per Advent of Code guidelines

You may find the puzzle description at: https://adventofcode.com/2021/day/6

In [11]:
// var part2TestAnswer = CountAllFish(testInputLines, daysRemain: 256);
// Console.WriteLine(part2TestAnswer);

Ok, unsurpsisingly, it looks like cranking up part 1's solution to 256 days doesn't work. Perhaps there is some mathematical shortcut, but I can't see it since the "replication interval" is not constant. But our recursive solution is essentially a depth-search "search", so I think we can cache the results and hopefully that gives us the speed we need.

In [12]:
using FishT = long; // int overflows
using FishKey = (int n, int daysRemain);

FishT NewFish2(int n, int daysRemain)
{
    Dictionary<FishKey, FishT> fishCache = new();

    return NewFishInner(n, daysRemain);

    FishT NewFishInner(int n, int daysRemain)
    {
        if (daysRemain < 0) { return 0; }

        var cacheKey = (n, daysRemain);
        if (fishCache.TryGetValue(cacheKey, out var cachedResult))
        {
            return cachedResult;
        };

        FishT result = 1;
        while (daysRemain > 0)
        {
            daysRemain -= (n + 1);
            n = 6;

            if (daysRemain >= 0)
            {
                result += NewFishInner(8, daysRemain);
            }
        }

        fishCache[cacheKey] = result;
        return result;
    }
}

In [13]:
FishT CountAllFish2(string[] inputLines)
{
    var nums = inputLines[0].ParseInts().ToArray();

    var result = nums.Select(n => NewFish2(n, daysRemain: 256)).Sum();
    return result;
}

In [14]:
// After 256 days in the example above, there would be a total of 26984457539
// lanternfish!

var part2TestAnswer = CountAllFish2(testInputLines);
Console.WriteLine(part2TestAnswer);

26984457539


In [15]:
// How many lanternfish would there be after 256 days?

var part2Answer = CountAllFish2(inputLines);
Console.WriteLine(part2Answer);

1574445493136


In [16]:
// 1574445493136 is correct!
Ensure(1574445493136, part2Answer);

### Alternative approach

Looking at [Peter Norvig's approach](https://github.com/norvig/pytudes/blob/main/ipynb/Advent-2021.ipynb), he has as-usual provided an elegant solution: maintaining a count of fish per remaining day, and migrating the fish each day, let's try it!

In [17]:
public FishT CountAllFish3(string[] inputLines, int days)
{
    var initialFish = inputLines[0].ParseInts().ToList();
    var fishPerDay = new FishT[9];
    initialFish.ForEach(n => fishPerDay[n]++);

    foreach (var _ in Range(0, days))
    {
        var tomorrowFishPerDay = new FishT[9];

        foreach (var i in Range(0, fishPerDay.Length))
        {
            var todayFish = fishPerDay[i];

            // Move today's count to tomorrow. For those fish on 0, move them
            // back to 6, and also add their offspring into day 8
            var tmrw = i is 0 ? 6 : i - 1;
            tomorrowFishPerDay[tmrw] += todayFish;
            tomorrowFishPerDay[8] += i is 0 ? todayFish : 0;
        }
        
        fishPerDay = tomorrowFishPerDay;
    }

    return fishPerDay.Sum();
}

var altTestResult = CountAllFish3(testInputLines, days: 256);
Console.WriteLine(altTestResult);

var altResult = CountAllFish3(inputLines, days: 256);
Console.WriteLine(altResult);
Ensure(1574445493136, altResult);


26984457539
1574445493136
